In [1]:
# Week 3 Day 4 — CNN Fundamentals
# Goal: Understand filters and convolution on CT images

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("✓ Setup complete")

Device: cuda
✓ Setup complete


In [2]:
# Cell 2 — Manual convolution
# Sliding filter across CT patch manually

print("── CT patch ──")
# Small CT patch — 5×5 HU values
ct_patch = torch.tensor([
    [0.,    0.,    0.,    0.,    0.],
    [0.,    0.,    0.,    0.,    0.],
    [0.,  100.,  100.,  100.,    0.],
    [0.,  100.,  100.,  100.,    0.],
    [0.,    0.,    0.,    0.,    0.]
])
print(ct_patch)
print(f"CT patch shape: {ct_patch.shape}")

print("\n── Edge detection filter ──")
# This filter detects horizontal edges
edge_filter = torch.tensor([
    [-1., -1., -1.],
    [ 0.,  0.,  0.],
    [ 1.,  1.,  1.]
])
print(edge_filter)

print("\n── Manual convolution at one position ──")
# Apply filter to top-left 3×3 patch
patch_3x3 = ct_patch[0:3, 0:3]
print(f"Image patch:\n{patch_3x3}")
print(f"Filter:\n{edge_filter}")

# Multiply and sum
result = (patch_3x3 * edge_filter).sum()
print(f"\nMultiply and sum: {result}")
print(f"Output value: {result.item()}")

print("\n── What this means ──")
print("Large value → strong edge detected here")
print("Zero value  → no edge at this location")

── CT patch ──
tensor([[  0.,   0.,   0.,   0.,   0.],
        [  0.,   0.,   0.,   0.,   0.],
        [  0., 100., 100., 100.,   0.],
        [  0., 100., 100., 100.,   0.],
        [  0.,   0.,   0.,   0.,   0.]])
CT patch shape: torch.Size([5, 5])

── Edge detection filter ──
tensor([[-1., -1., -1.],
        [ 0.,  0.,  0.],
        [ 1.,  1.,  1.]])

── Manual convolution at one position ──
Image patch:
tensor([[  0.,   0.,   0.],
        [  0.,   0.,   0.],
        [  0., 100., 100.]])
Filter:
tensor([[-1., -1., -1.],
        [ 0.,  0.,  0.],
        [ 1.,  1.,  1.]])

Multiply and sum: 200.0
Output value: 200.0

── What this means ──
Large value → strong edge detected here
Zero value  → no edge at this location


In [3]:
# Cell 3 — PyTorch convolution
# Using nn.Conv2d instead of manual calculation

import torch
import torch.nn as nn

print("── nn.Conv2d ──")

# Create convolution layer
# in_channels=1  → CT has 1 channel
# out_channels=1 → one filter
# kernel_size=3  → 3×3 filter
conv = nn.Conv2d(in_channels=1,
                 out_channels=1,
                 kernel_size=3,
                 padding=1)    # padding keeps same size

print(f"Conv layer created")
print(f"Filter shape: {conv.weight.shape}")

# Prepare CT patch for PyTorch
# needs shape [batch, channel, height, width]
ct_patch = torch.tensor([
    [0.,    0.,    0.,    0.,    0.],
    [0.,    0.,    0.,    0.,    0.],
    [0.,  100.,  100.,  100.,    0.],
    [0.,  100.,  100.,  100.,    0.],
    [0.,    0.,    0.,    0.,    0.]
])

# Add batch and channel dimensions
ct_input = ct_patch.unsqueeze(0).unsqueeze(0)
print(f"\nInput shape: {ct_input.shape}")

# Apply convolution
with torch.no_grad():
    output = conv(ct_input)
print(f"Output shape: {output.shape}")

# Set our edge detection filter manually
with torch.no_grad():
    conv.weight[0, 0] = torch.tensor([
        [-1., -1., -1.],
        [ 0.,  0.,  0.],
        [ 1.,  1.,  1.]
    ])
    conv.bias[0] = 0.0

# Apply edge detection
with torch.no_grad():
    edge_output = conv(ct_input)

print(f"\nEdge detection output:")
print(edge_output[0, 0])
print("\nLarge values = edges detected")
print("Zero values  = no edge")

── nn.Conv2d ──
Conv layer created
Filter shape: torch.Size([1, 1, 3, 3])

Input shape: torch.Size([1, 1, 5, 5])
Output shape: torch.Size([1, 1, 5, 5])

Edge detection output:
tensor([[   0.,    0.,    0.,    0.,    0.],
        [ 100.,  200.,  300.,  200.,  100.],
        [ 100.,  200.,  300.,  200.,  100.],
        [-100., -200., -300., -200., -100.],
        [-100., -200., -300., -200., -100.]])

Large values = edges detected
Zero values  = no edge


In [1]:
# Cell 4 — Max Pooling

import torch
import torch.nn as nn

print("── Max Pooling ──")
print("Takes maximum value from each 2×2 patch")
print("Reduces feature map size by half")

# Create max pooling layer
pool = nn.MaxPool2d(kernel_size=2, stride=2)

print("\n── Simple example ──")
# Small 4×4 feature map
feature_map = torch.tensor([[
    [ 10.,   2.,   8.,   1.],
    [  3.,  15.,   4.,   9.],
    [  7.,   1.,  12.,   3.],
    [  2.,   8.,   5.,  11.]
]])

# Add batch dimension
feature_map = feature_map.unsqueeze(0)
print(f"Input shape:  {feature_map.shape}")
print(f"Input:\n{feature_map[0,0]}")

# Apply max pooling
output = pool(feature_map)
print(f"\nOutput shape: {output.shape}")
print(f"Output:\n{output[0,0]}")

print("\n── Explanation ──")
print("Patch 1 (top left):    10, 2, 3, 15  → max = 15")
print("Patch 2 (top right):    8, 1, 4,  9  → max =  9")
print("Patch 3 (bottom left):  7, 1, 2,  8  → max =  8")
print("Patch 4 (bottom right):12, 3, 5, 11  → max = 12")

print("\n── On CT edge output ──")
# Use edge output from Cell 3
edge_map = torch.tensor([[
    [   0.,    0.,    0.,    0.,    0.],
    [ 100.,  200.,  300.,  200.,  100.],
    [ 100.,  200.,  300.,  200.,  100.],
    [-100., -200., -300., -200., -100.],
    [-100., -200., -300., -200., -100.]
]]).unsqueeze(0)

print(f"Edge map shape:  {edge_map.shape}")
output = pool(edge_map)
print(f"After pooling:   {output.shape}")
print(f"Output:\n{output[0,0]}")

── Max Pooling ──
Takes maximum value from each 2×2 patch
Reduces feature map size by half

── Simple example ──
Input shape:  torch.Size([1, 1, 4, 4])
Input:
tensor([[10.,  2.,  8.,  1.],
        [ 3., 15.,  4.,  9.],
        [ 7.,  1., 12.,  3.],
        [ 2.,  8.,  5., 11.]])

Output shape: torch.Size([1, 1, 2, 2])
Output:
tensor([[15.,  9.],
        [ 8., 12.]])

── Explanation ──
Patch 1 (top left):    10, 2, 3, 15  → max = 15
Patch 2 (top right):    8, 1, 4,  9  → max =  9
Patch 3 (bottom left):  7, 1, 2,  8  → max =  8
Patch 4 (bottom right):12, 3, 5, 11  → max = 12

── On CT edge output ──
Edge map shape:  torch.Size([1, 1, 5, 5])
After pooling:   torch.Size([1, 1, 2, 2])
Output:
tensor([[200., 300.],
        [200., 300.]])


In [2]:
# Cell 5 — Simple CNN
# Combining convolution + ReLU + pooling

import torch
import torch.nn as nn

print("── Building Simple CNN ──")

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Layer 1: convolution
        self.conv1 = nn.Conv2d(in_channels=1,
                               out_channels=4,
                               kernel_size=3,
                               padding=1)
        # Activation
        self.relu = nn.ReLU()
        # Pooling
        self.pool = nn.MaxPool2d(kernel_size=2,
                                 stride=2)
        # Layer 2: convolution
        self.conv2 = nn.Conv2d(in_channels=4,
                               out_channels=8,
                               kernel_size=3,
                               padding=1)

    def forward(self, x):
        print(f"Input:          {x.shape}")

        # Conv1 + ReLU + Pool
        x = self.conv1(x)
        print(f"After conv1:    {x.shape}")
        x = self.relu(x)
        print(f"After ReLU:     {x.shape}")
        x = self.pool(x)
        print(f"After pooling:  {x.shape}")

        # Conv2 + ReLU + Pool
        x = self.conv2(x)
        print(f"After conv2:    {x.shape}")
        x = self.relu(x)
        print(f"After ReLU:     {x.shape}")
        x = self.pool(x)
        print(f"After pooling:  {x.shape}")

        return x

# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN().to(device)

# Simulate CT slice
ct_input = torch.zeros(1, 1, 64, 64).to(device)
print(f"Device: {device}")
print()

# Forward pass
with torch.no_grad():
    output = model(ct_input)

print(f"\nFinal output shape: {output.shape}")
print(f"\n── Shape journey ──")
print(f"Input:      [1, 1, 64, 64]")
print(f"→ conv1:    [1, 4, 64, 64]  filters: 1→4")
print(f"→ pool:     [1, 4, 32, 32]  size: 64→32")
print(f"→ conv2:    [1, 8, 32, 32]  filters: 4→8")
print(f"→ pool:     [1, 8, 16, 16]  size: 32→16")

── Building Simple CNN ──
Device: cuda

Input:          torch.Size([1, 1, 64, 64])
After conv1:    torch.Size([1, 4, 64, 64])
After ReLU:     torch.Size([1, 4, 64, 64])
After pooling:  torch.Size([1, 4, 32, 32])
After conv2:    torch.Size([1, 8, 32, 32])
After ReLU:     torch.Size([1, 8, 32, 32])
After pooling:  torch.Size([1, 8, 16, 16])

Final output shape: torch.Size([1, 8, 16, 16])

── Shape journey ──
Input:      [1, 1, 64, 64]
→ conv1:    [1, 4, 64, 64]  filters: 1→4
→ pool:     [1, 4, 32, 32]  size: 64→32
→ conv2:    [1, 8, 32, 32]  filters: 4→8
→ pool:     [1, 8, 16, 16]  size: 32→16
